# Relative PSF-Flux Light Curves per DIA Object — src + forced photometry

This notebook extends `04_relativeFlux.ipynb` by overlaying **forced-photometry (fp)** points
on top of the src (alert) relative-flux light curves.

Only **psfFlux** is used (apFlux is ignored).

### Data sources
| File | Content |
|---|---|
| `data_FINK_BLOCK_LC_03/src_joined_butler.parquet` (or `consdb`) | Alert-stream sources (src) |
| `data_FINK_BLOCK_LC_01/gaia_star_stable_fp.parquet` | Forced photometry (fp) |

### Normalisation convention
The normalisation factor for each *(diaObjectId, band)* pair is the **median of the src psfFlux**.
The fp points are divided by the same median.  If a diaObjectId has no src in a given band,
the band is skipped for that object (no median available).  If there are no fp rows for an object
or a band, fp points are simply absent — this is handled gracefully.

### Visual convention
| Data type | Marker | Fill |
|---|---|---|
| src | filled circle `o` | band colour |
| fp  | circle `o` | **hollow** (no fill) — edge colour = band colour |

Subplot layout: `u | g | r | i | z | y | all-bands`  (7 panels per figure).

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Last update:** 2026-04-02  
- **Subject:** Fink alert broker — Rubin LSST photometric calibration check (src + fp)

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_04c"
DIR_DATA_SRC = "data_FINK_BLOCK_LC_03"  # src alert data (from notebook 03)
DIR_DATA_FP = "data_FINK_BLOCK_LC_01"  # forced photometry data (from notebook 01)
DIR_FIGS = f"figs_{NB_TAG}"  # output figures
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files (src) ────────────────────────────────────────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_SRC, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_SRC, "src_joined_consdb.parquet")

# ── Forced photometry file ────────────────────────────────────────────────────
FILE_FP = os.path.join(DIR_DATA_FP, "gaia_star_stable_fp.parquet")

# ── Number of top-ranked DIA objects to plot ──────────────────────────────────
N_TOP = 20  # <── change here

# ── Band definitions ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names in the src parquet ──────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"

# ── Column names in the fp parquet ───────────────────────────────────────────
# Adjust these if the fp file uses different column names
FP_COL_OBJ = "diaObjectId"
FP_COL_MJD = "midpointMjdTai"
FP_COL_BAND = "band"
FP_COL_PSF = "psfFlux"
FP_COL_PSFERR = "psfFluxErr"


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Src input directory : {os.path.abspath(DIR_DATA_SRC)}")
print(f"FP  input directory : {os.path.abspath(DIR_DATA_FP)}")
print(f"Figure directory    : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP               : {N_TOP}")

## 2. Load src data (alert stream)

In [ ]:
# Butler preferred; fallback to consDb
if os.path.exists(FILE_BUTLER):
    df_src = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file: {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df_src = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df_src.shape[0]:,} rows × {df_src.shape[1]} columns")
print(f"Source label: {src_label}")

## 3. Load forced photometry data

In [ ]:
if not os.path.exists(FILE_FP):
    raise FileNotFoundError(
        f"Forced photometry file not found: {FILE_FP}\n"
        "Run notebook 01 to produce gaia_star_stable_fp.parquet first."
    )

df_fp_all = pd.read_parquet(FILE_FP)
print(f"Forced photometry file: {FILE_FP}")
print(f"Shape: {df_fp_all.shape[0]:,} rows × {df_fp_all.shape[1]} columns")

## 4. Schema inspection

In [ ]:
print("=== SRC columns ===")
print(df_src.dtypes.to_string())

In [ ]:
print("=== FP columns ===")
print(df_fp_all.dtypes.to_string())

In [ ]:
# Validate required src columns
required_src = [COL_OBJ, COL_MJD, COL_BAND, COL_PSF, COL_PSFERR]
missing_src = [c for c in required_src if c not in df_src.columns]
if missing_src:
    raise KeyError(f"Missing required src columns: {missing_src}")
print("All required src columns present.")

# Auto-detect fp column names if they differ from defaults
# (the fp parquet might use the same 'r:' prefix or plain names)
fp_cols = df_fp_all.columns.tolist()


def _find_col(candidates, available, label):
    for c in candidates:
        if c in available:
            return c
    raise KeyError(
        f"Cannot find {label} column in fp file. Candidates tried: {candidates}. Available: {available[:20]}"
    )


FP_COL_OBJ = _find_col(["diaObjectId", "r:diaObjectId", "objectId"], fp_cols, "diaObjectId")
FP_COL_MJD = _find_col(["midpointMjdTai", "r:midpointMjdTai", "mjd", "MJD"], fp_cols, "midpointMjdTai")
FP_COL_BAND = _find_col(["band", "r:band", "filter"], fp_cols, "band")
FP_COL_PSF = _find_col(["psfFlux", "r:psfFlux"], fp_cols, "psfFlux")
FP_COL_PSFERR = _find_col(["psfFluxErr", "r:psfFluxErr"], fp_cols, "psfFluxErr")

print(f"FP column mapping:")
print(f"  diaObjectId   → {FP_COL_OBJ}")
print(f"  midpointMjdTai→ {FP_COL_MJD}")
print(f"  band          → {FP_COL_BAND}")
print(f"  psfFlux       → {FP_COL_PSF}")
print(f"  psfFluxErr    → {FP_COL_PSFERR}")

## 5. Rank DIA objects by number of src alerts (decreasing)

In [ ]:
# Count src alerts per diaObjectId and sort descending
alert_counts = df_src.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)

print(f"Total unique DIA objects in src : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
# Select top-N objects
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df_src[df_src[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)

# Filter fp to top-N objects
# fp uses its own object id column (may be int or str — match type)
fp_obj_ids = df_fp_all[FP_COL_OBJ].unique()
# Attempt type coercion for safe intersection
try:
    top_objects_fp = [type(fp_obj_ids[0])(x) for x in top_objects]
except Exception:
    top_objects_fp = top_objects

df_fp = df_fp_all[df_fp_all[FP_COL_OBJ].isin(top_objects_fp)].copy()
df_fp[FP_COL_MJD] = df_fp[FP_COL_MJD].astype(float)

print(f"Src rows for top-{N_TOP} objects : {len(df_top):,}")
print(f"FP  rows for top-{N_TOP} objects : {len(df_fp):,}")
print(f"diaObjectIds in fp (top-N) : {df_fp[FP_COL_OBJ].nunique()}")

## 6. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """
    Convert an array of MJD (float) values to ISO date strings 'YYYY-MM-DD'.
    Uses astropy.time.Time for accuracy.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_src_median_and_ratio(flux, flux_err):
    """
    Compute flux / median(flux) and propagate errors.

    The median is computed on finite positive values.

    Parameters
    ----------
    flux     : array-like  — measured fluxes (nJy)
    flux_err : array-like  — 1-sigma flux uncertainties (nJy)

    Returns
    -------
    ratio      : ndarray   — flux / median(flux)
    ratio_err  : ndarray   — propagated 1-sigma uncertainty
    f_med      : float     — the median flux used for normalisation
    sigma_ratio: float     — RMS scatter of the ratio
    """
    f = np.asarray(flux, dtype=float)
    fe = np.asarray(flux_err, dtype=float)

    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), np.nan, np.nan

    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), 0.0, np.nan

    ratio = f / f_med
    ratio_err = fe / f_med
    sigma_ratio = float(np.nanstd(ratio[finite_mask]))

    return ratio, ratio_err, f_med, sigma_ratio


def normalise_fp(fp_flux, fp_flux_err, f_med_src):
    """
    Normalise fp fluxes using the src median.

    Parameters
    ----------
    fp_flux     : array-like — fp psfFlux values
    fp_flux_err : array-like — fp psfFluxErr values
    f_med_src   : float      — median of src psfFlux for same (object, band)

    Returns
    -------
    ratio     : ndarray
    ratio_err : ndarray
    """
    f = np.asarray(fp_flux, dtype=float)
    fe = np.asarray(fp_flux_err, dtype=float)

    if np.isnan(f_med_src) or f_med_src == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan)

    return f / f_med_src, fe / f_med_src


def _add_date_axis(ax, dt_array, t0_mjd):
    """
    Add a secondary x-axis on top of *ax* showing calendar dates.

    Parameters
    ----------
    ax       : matplotlib Axes
    dt_array : 1-D array of Δt [days] already plotted on ax
    t0_mjd   : float  — reference MJD (global first alert)
    """
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return

    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        n_ticks = 1
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)

    tick_mjd = t0_mjd + tick_dt
    tick_dates = mjd_to_datestr(tick_mjd)

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


print("Utility functions defined.")

## 7. Main plotting function: src + fp relative PSF-flux light curves

For each DIA object:
- 7 subplots: one per band `u g r i z y` + one combined panel (subplot 7)
- **src** plotted as filled circles `o` (band colour)
- **fp**  plotted as hollow circles `o` with edge colour = band colour, no fill
- Normalisation by `median(src psfFlux)` per *(object, band)*
- x bottom : Δt [days] from first src alert; x top : calendar dates

In [ ]:
def plot_src_fp_object(obj_id, df_src_obj, df_fp_obj, axes):
    """
    Plot relative PSF-flux light curves (src + fp) for one DIA object.

    Parameters
    ----------
    obj_id     : int / str  — diaObjectId
    df_src_obj : DataFrame  — src rows for this object only
    df_fp_obj  : DataFrame  — fp rows for this object (may be empty)
    axes       : list of 7 Axes — [u, g, r, i, z, y, all-bands]

    Returns
    -------
    t0_date : str  — ISO date of the first src alert
    """
    # Global t0 = MJD of the very first src alert across all bands
    t0_mjd = df_src_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    # Collect all Δt values (src + fp) for the date-axis range
    all_dt_src = df_src_obj[COL_MJD].values - t0_mjd
    all_dt_fp = (df_fp_obj[FP_COL_MJD].values - t0_mjd) if len(df_fp_obj) > 0 else np.array([])
    all_dt_combined = np.concatenate([all_dt_src, all_dt_fp])

    # Store per-band median of src (needed for fp normalisation)
    src_median_per_band = {}

    # ── Per-band subplots (subplots 0..5) ─────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        # --- SRC ---
        mask_src = df_src_obj[COL_BAND] == band
        df_b_src = df_src_obj[mask_src].sort_values(COL_MJD)

        if len(df_b_src) == 0:
            # No src in this band → cannot normalise → skip band entirely
            src_median_per_band[band] = np.nan
            ax.text(
                0.5,
                0.5,
                "no src data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            continue

        dt_src = df_b_src[COL_MJD].values - t0_mjd
        src_ratio, src_ratio_err, f_med, sigma_src = compute_src_median_and_ratio(
            df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
        )
        src_median_per_band[band] = f_med

        ax.errorbar(
            dt_src,
            src_ratio,
            yerr=src_ratio_err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.8,
            capsize=2,
            alpha=0.85,
            label=f"src  σ={sigma_src:.3f}",
        )

        # --- FP ---
        if len(df_fp_obj) > 0:
            mask_fp = df_fp_obj[FP_COL_BAND] == band
            df_b_fp = df_fp_obj[mask_fp].sort_values(FP_COL_MJD)

            if len(df_b_fp) > 0 and not np.isnan(f_med):
                dt_fp = df_b_fp[FP_COL_MJD].values - t0_mjd
                fp_ratio, fp_ratio_err = normalise_fp(
                    df_b_fp[FP_COL_PSF].values,
                    df_b_fp[FP_COL_PSFERR].values,
                    f_med,
                )
                fp_sigma = float(np.nanstd(fp_ratio[np.isfinite(fp_ratio)]))
                ax.errorbar(
                    dt_fp,
                    fp_ratio,
                    yerr=fp_ratio_err,
                    fmt="o",
                    ms=5,
                    color="none",  # hollow fill
                    mec=color,  # edge colour = band colour
                    mew=1.2,
                    ecolor=color,
                    elinewidth=0.8,
                    capsize=2,
                    alpha=0.80,
                    label=f"fp   σ={fp_sigma:.3f}",
                )

        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}  |  σ_src={sigma_src:.3f}", fontsize=9)
        ax.set_ylabel("psfFlux / median(src)", fontsize=8)
        ax.legend(fontsize=7, loc="best")

        # Date axis — use combined src+fp range for this band
        dt_all_band = dt_src
        if len(df_fp_obj) > 0:
            mask_fp = df_fp_obj[FP_COL_BAND] == band
            if mask_fp.sum() > 0:
                dt_fp_band = df_fp_obj[mask_fp][FP_COL_MJD].values - t0_mjd
                dt_all_band = np.concatenate([dt_src, dt_fp_band])
        _add_date_axis(ax, dt_all_band, t0_mjd)

    # ── Subplot 7: all bands combined ──────────────────────────────────────────
    ax_all = axes[-1]

    for band in BANDS:
        color = BAND_COLORS.get(band, "k")
        f_med = src_median_per_band.get(band, np.nan)

        # src
        mask_src = df_src_obj[COL_BAND] == band
        df_b_src = df_src_obj[mask_src].sort_values(COL_MJD)
        if len(df_b_src) > 0 and not np.isnan(f_med):
            dt_src = df_b_src[COL_MJD].values - t0_mjd
            src_ratio, src_ratio_err, _, sigma_src = compute_src_median_and_ratio(
                df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
            )
            ax_all.errorbar(
                dt_src,
                src_ratio,
                yerr=src_ratio_err,
                fmt="o",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.75,
                label=f"{band} src σ={sigma_src:.3f}",
            )

        # fp
        if len(df_fp_obj) > 0 and not np.isnan(f_med):
            mask_fp = df_fp_obj[FP_COL_BAND] == band
            df_b_fp = df_fp_obj[mask_fp].sort_values(FP_COL_MJD)
            if len(df_b_fp) > 0:
                dt_fp = df_b_fp[FP_COL_MJD].values - t0_mjd
                fp_ratio, fp_ratio_err = normalise_fp(
                    df_b_fp[FP_COL_PSF].values,
                    df_b_fp[FP_COL_PSFERR].values,
                    f_med,
                )
                ax_all.errorbar(
                    dt_fp,
                    fp_ratio,
                    yerr=fp_ratio_err,
                    fmt="o",
                    ms=4,
                    color="none",
                    mec=color,
                    mew=1.2,
                    ecolor=color,
                    elinewidth=0.7,
                    capsize=2,
                    alpha=0.70,
                    label=f"{band} fp",
                )

    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands — src (●) + fp (○)", fontsize=9)
    ax_all.set_ylabel("psfFlux / median(src)", fontsize=8)
    ax_all.legend(fontsize=6, ncol=3, loc="best")
    _add_date_axis(ax_all, all_dt_combined, t0_mjd)

    return t0_date


print("plot_src_fp_object() defined.")

## 8. Main loop: plot all top-N objects

In [ ]:
n_subplots = len(BANDS) + 1  # 6 bands + combined (subplot 7)

for rank, obj_id in enumerate(top_objects):
    # --- src rows for this object ---
    df_src_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_src_obj)

    # --- fp rows for this object (may be empty) ---
    # Match type robustly
    try:
        fp_obj_id = type(df_fp_all[FP_COL_OBJ].iloc[0])(obj_id)
    except Exception:
        fp_obj_id = obj_id
    df_fp_obj = df_fp[df_fp[FP_COL_OBJ] == fp_obj_id].sort_values(FP_COL_MJD)
    n_fp = len(df_fp_obj)

    # Retrieve field name (first value; might be a Series — take unique string)
    field_vals = df_src_obj["field"].dropna().unique() if "field" in df_src_obj.columns else []
    field_str = field_vals[0] if len(field_vals) > 0 else "?"

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        sharey=False,
        constrained_layout=True,
    )

    t0_date = plot_src_fp_object(obj_id, df_src_obj, df_fp_obj, axes)

    # Common bottom x-label
    for ax in axes:
        ax.set_xlabel("Δt [days] from first src alert", fontsize=8)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field_str}  |  "
        f"N_src={n_alerts}  N_fp={n_fp}  |  t₀={t0_date}\n"
        "src ● (filled)   fp ○ (hollow)   — normalised by median(src psfFlux)",
        fontsize=10,
        y=1.05,
    )

    savefig(f"relflux_psf_srcfp_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()
    # plt.close(fig)

## 9. Summary table: per-object, per-band σ(ratio)

Reports the RMS scatter of the src relative flux ratio per object and band.
Also indicates how many fp points are available per *(object, band)*.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_src_obj = df_top[df_top[COL_OBJ] == obj_id]
    n_total = len(df_src_obj)
    t0_mjd = df_src_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    try:
        fp_obj_id = type(df_fp_all[FP_COL_OBJ].iloc[0])(obj_id)
    except Exception:
        fp_obj_id = obj_id
    df_fp_obj = df_fp[df_fp[FP_COL_OBJ] == fp_obj_id]

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_src": n_total,
        "n_fp": len(df_fp_obj),
        "t0_date": t0_date,
    }

    for band in BANDS:
        df_b_src = df_src_obj[df_src_obj[COL_BAND] == band]
        if len(df_b_src) == 0:
            row[f"sigma_src_{band}"] = np.nan
            row[f"n_fp_{band}"] = 0
            continue

        _, _, _, sigma_src = compute_src_median_and_ratio(
            df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
        )
        row[f"sigma_src_{band}"] = round(sigma_src, 4)

        n_fp_band = int((df_fp_obj[FP_COL_BAND] == band).sum()) if len(df_fp_obj) > 0 else 0
        row[f"n_fp_{band}"] = n_fp_band

    records.append(row)

df_summary = pd.DataFrame(records)
print("Summary table — relative-flux scatter (src) + fp counts per object & band:")
display(df_summary)

In [ ]:
# Save summary table
summary_path = os.path.join(DIR_FIGS, f"sigma_ratio_srcfp_summary_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")